In [1]:
import sys
from pathlib import Path
from torch.optim import AdamW
import torch
from torch.utils.data import Dataset, DataLoader
import open_clip
import matplotlib.pyplot as plt
import lpips

In [2]:
sys.path.append(str(Path().resolve().parent))
str(Path().resolve().parent)

'/Users/mohamedmafaz/Desktop/img2img_DiT'

In [3]:
from DiT_model import DiT
from data import img_dataset
from VAE import VAE
from DDPM import LinearNoiseScheduler

In [4]:
import json
with open("../config.json", "r") as file:
    config = json.load(file)

device = "cpu"

In [5]:
vae = VAE(device = device, freeze = True, scaling_factor = config["VAE"]["scaling_factor"], path = "../sd-vae-ft-mse").to(device)

dit = DiT(d_model           = config["DiT"]["d_model"],
          g_channels        = config["DiT"]["g_channels"],
          grid_size         = config["DiT"]["grid_size"],
          patch_size        = config["DiT"]["patch_size"],
          timestep_emb_dim  = config["DiT"]["timestep_emb_dim"],
          number_emb_dim    = config["DiT"]["number_emb_dim"],
          num_layers        = config["DiT"]["num_layers"],
          num_heads         = config["DiT"]["num_heads"],
          CLIP_dim          = config["DiT"]["CLIP_dim"])

scheduler = LinearNoiseScheduler(num_timesteps  = config["Scheduler"]["num_timesteps"],
                                     beta_start = config["Scheduler"]["beta_start"],
                                     beta_end   = config["Scheduler"]["beta_end"])

In [6]:
MSE_loss_fn = torch.nn.MSELoss()
clip_model = open_clip.create_model(
    "ViT-L-14",
    pretrained="../models--timm--vit_large_patch14_clip_224.openai/snapshots/18d0535469bb561bf468d76c1d73aa35156c922b/open_clip_model.safetensors"
)
lpips_loss_fn = lpips.LPIPS(net='vgg', model_path = "../vgg.pth", verbose = False)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
optimizer = AdamW(dit.parameters(), lr=config["Training"]["learning_rate"], weight_decay=0)

In [8]:
dataset = img_dataset(root_dir = "../cars", img_size = 224)
dataloader = DataLoader(dataset, batch_size = 1, shuffle = True)

In [9]:
for param in vae.parameters():
    param.requires_grad = False

for param in clip_model.parameters():
    param.requires_grad = False

In [10]:
for condition_image, actual_image in dataloader:
    break

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torchvision/transforms/functional.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  img = torch.from_numpy(pic.transpose((2, 0, 1))).contiguous()


In [11]:
condition_image = condition_image.to(device)
actual_image    = actual_image.float().to(device)

# MSE DiT Loss

In [12]:
# actual image VAE-ing
with torch.no_grad():
    mu, logvar = vae.encode(actual_image)
    z = vae.reparameterize(mu, logvar)   # [B, 4, 28, 28]

# condition image VAE-ing
with torch.no_grad():
    mu_c, logvar_c = vae.encode(condition_image)
    z_c = vae.reparameterize(mu_c, logvar_c) # [B, 4, 28, 28]

# Sample random noise
noise = torch.randn_like(z).to(device)   # [B, 4, 28, 28]

B = z.shape[0]
# Sample timestep
t = torch.randint(0, 1000,(B,)).to(device) # [B]

# adding nooise to actual image
noisy_im = scheduler.add_noise(z, noise, t) # [B, 4, 28, 28]

# CLIP encoding
clip_condition = clip_model.encode_image(condition_image) # [B, 768]

# DiT Prediction (in latent space)
dit_pred = dit(noisy_im, t, z_c, clip_condition)

diffusion_mse_loss = MSE_loss_fn(dit_pred, noise)
diffusion_mse_loss

tensor(1.0495, grad_fn=<MseLossBackward0>)

# CLIP loss

In [13]:
# CLIP encoding for actual image
with torch.no_grad():
    target_emb = clip_model.encode_image(actual_image)

t_clip = torch.randint(800, 1000, (B,)).to(device)
# adding nooise to actual image
noisy_im = scheduler.add_noise(z, noise, t_clip) # [B, 4, 28, 28]

# DiT Prediction (in latent space)
dit_pred = dit(noisy_im, t_clip, z_c, clip_condition)

x0_pred = scheduler.get_x0(noisy_im, dit_pred, t_clip)

# decoding image to pixel space
predicted_image = vae.decode(x0_pred)

# encoding the predicted image
pred_emb = clip_model.encode_image(predicted_image)

pred_emb   = pred_emb / pred_emb.norm(dim=-1, keepdim=True)
target_emb = target_emb / target_emb.norm(dim=-1, keepdim=True)

clip_loss = 1 - (pred_emb * target_emb).sum(dim=-1).mean()   # 1 - cosine similarity
clip_loss

tensor(0.4338, grad_fn=<RsubBackward1>)

# LPIPS loss

In [23]:
lpips_loss = lpips_loss_fn(predicted_image, actual_image).sum(dim=0).mean()
lpips_loss

tensor(0.7455, grad_fn=<MeanBackward0>)

# Training loop

In [24]:
for param in vae.parameters():
    param.requires_grad = False

for param in clip_model.parameters():
    param.requires_grad = False

In [ ]:
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1,3,1,1)
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1,3,1,1)

def clip_preproces(img):
    # images are normalised [-1, 1] for VAE and DiT
    img = (img + 1)/2
    if (img.shape[-1] != 224) or (img.shape[-2] != 224):
        img = torch.nn.functional.interpolate(img, size=224, mode='bicubic', align_corners=False)
    img = (img - CLIP_MEAN) / CLIP_STD
    return img

In [ ]:
losses = []
start_epoch = 0

lambda_diffuion = config["weights"]["diffusion"]
lambda_clip     = config["weights"]["CLIP"]
lambda_lpips    = config["weights"]["lpips"]
acc_steps = config["Training"]["accumulation_step"]

epochs = config["Training"]["epochs"]

for epoch in range(start_epoch, epochs):
    epoch_loss = 0.0
    step_count = 0
    for condition_image, actual_image in dataloader:
        step_count += 1

        condition_image = condition_image.to(device)
        actual_image    = actual_image.float().to(device)

        ######################
        # DIFFUSION MSE LOSS #
        ######################

        # actual image VAE-ing
        with torch.no_grad():
            mu, logvar = vae.encode(actual_image)
            z = vae.reparameterize(mu, logvar)   # [B, 4, 28, 28]

        # condition image VAE-ing
        with torch.no_grad():
            mu_c, logvar_c = vae.encode(condition_image)
            z_c = vae.reparameterize(mu_c, logvar_c) # [B, 4, 28, 28]

        # Sample random noise
        noise = torch.randn_like(z).to(device)   # [B, 4, 28, 28]

        B = z.shape[0]
        # Sample timestep
        t = torch.randint(0, 1000,(B,)).to(device) # [B]

        # adding nooise to actual image
        noisy_im = scheduler.add_noise(z, noise, t) # [B, 4, 28, 28]

        # CLIP encoding
        clip_condition = clip_model.encode_image(clip_preproces(condition_image)) # [B, 768]

        # DiT Prediction (in latent space)
        dit_pred = dit(noisy_im, t, z_c, clip_condition)

        diffusion_mse_loss = MSE_loss_fn(dit_pred, noise)


        #############
        # CLIP LOSS #
        #############

        # CLIP encoding for actual image
        with torch.no_grad():
            target_emb = clip_model.encode_image(clip_preproces(actual_image))

        t_clip = torch.randint(0, 200, (B,)).to(device)
        # adding nooise to actual image
        noisy_im = scheduler.add_noise(z, noise, t_clip) # [B, 4, 28, 28]

        # DiT Prediction (in latent space)
        dit_pred = dit(noisy_im, t_clip, z_c, clip_condition)

        x0_pred = scheduler.get_x0(noisy_im, dit_pred, t_clip)

        # decoding image to pixel space
        predicted_image = vae.decode(x0_pred)

        # encoding the predicted image
        pred_emb = clip_model.encode_image(clip_preproces(predicted_image))

        pred_emb   = pred_emb / pred_emb.norm(dim=-1, keepdim=True)
        target_emb = target_emb / target_emb.norm(dim=-1, keepdim=True)

        clip_loss = 1 - (pred_emb * target_emb).sum(dim=-1).mean()   # 1 - cosine similarity

        ##############
        # LPIPS LOSS #
        ##############

        lpips_loss = lpips_loss_fn(predicted_image, actual_image).mean()


        ##############
        # TOTAL LOSS #
        ##############

        loss = (lambda_diffuion * diffusion_mse_loss) + (lambda_clip * clip_loss) + (lambda_lpips * lpips_loss)

        loss = loss / acc_steps
        loss.backward()
        if step_count % acc_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item()

    # flush the left over
    if step_count % acc_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    avg = (epoch_loss / len(dataloader)) * acc_steps
    losses.append(avg)

    print(f"[DiT] Epoch {epoch+1}/{epochs}  loss={avg:.6f}")
    
    if epoch % config["saves"]["DiT_Save_every"] == 0:
        torch.save(dit.state_dict(), config["saves"]["DiT_Path"])